# Diferencias finitas 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurimendiluce/AN2026/blob/main/diferencias_finitas/clase_1.ipynb)

### Problemas de valores de contorno: capa límite

Consideremos la ecuación

$$ \varepsilon U_{xx} - U_{x} = f, \qquad U(0)=\alpha, \quad U(1)=\beta. $$

Para el caso $f(x) = -1$, la solución exacta está dada por

$$ U_{\varepsilon}(x) = \alpha + x + (\beta - \alpha -1) \left( \frac{e^{x/\varepsilon}-1 }{e^{1/\varepsilon}-1} \right) $$

> Una ecuación de esta forma aparece, por ejemplo, al considerar el estado estacionario ($u_t=0$) de un problema de convección-difusión $u_t = \kappa u_{xx} + a u_x + \phi$, con constantes de difusividad $\kappa>0$ y de convección $a \in \mathbb{R}$. A la proporción $Pe = a/\kappa$ se la conoce como **número de Péclet**, y se toma $\varepsilon = 1/Pe$.

In [1]:
using LinearAlgebra, Plots

In [2]:
function U_ε(x,ε;α=1,β=3)
    y=α+x+(β-α-1)*((ℯ^(x/ε)-1)/(ℯ^(1/ε)-1))
    return y
end

U_ε (generic function with 1 method)

Graficamos la solución exacta para $\alpha=1,\ \beta=3$ a medida que $\varepsilon \to 0$:

In [ ]:
x=0:0.01:1

plot(x,U_ε.(x,0.3),label="epsilon=0.3")
plot!(x,U_ε.(x,0.1),label="epsilon=0.1")
plot!(x,U_ε.(x,0.05),label="epsilon=0.05")
plot!(x,U_ε.(x,0.01),label="epsilon=0.01")

#### ¿Por qué aparece la capa límite?

La ecuación $\varepsilon U_{xx} - U_x = f$ es un **problema de perturbación singular**: el parámetro pequeño $\varepsilon$ multiplica al término de mayor orden ($U_{xx}$). Esto tiene una consecuencia importante:

- Si tomamos $\varepsilon = 0$ directamente, la ecuación se reduce a una **ODE de primer orden**, $-U_x = f$ (la llamada **solución exterior**), que solo admite imponer **una** condición de borde, no dos.
- El término convectivo $-U_x$ transporta la información de izquierda a derecha, así que la condición compatible con esa dinámica es la de $x=0$: $U(0)=\alpha$. Con esa única condición la solución exterior queda determinada en $[0,1)$ — es la parte "lineal" que se ve lejos de $x=1$ en los gráficos de arriba.
- Como esa solución exterior en general **no** satisface $U(1)=\beta$, aparece cerca de $x=1$ una zona angosta — la **capa límite** — donde la solución se ajusta bruscamente para cumplir la condición de borde que faltaba.

Para estudiar esa zona se toma $\xi = (1-x)/\varepsilon$. Reescribiendo la ecuación en términos de $\xi$ y quedándonos lo que queda cuando $\varepsilon \to 0$, se obtiene que la corrección dentro de la capa decae como $e^{-\xi} = e^{-(1-x)/\varepsilon}$. Por eso el ancho característico de la capa límite es de orden $O(\varepsilon)$: cuanto más chico es $\varepsilon$, más angosta y abrupta es la transición, tal como se observó al graficar la solución exacta para $\varepsilon$ decreciente.

Para $f=-1$, $\alpha=1$, $\beta=3$ la ecuación reducida es $U_x=1$, cuya solución general es $U(x)=x+C$. Imponiendo $U(0)=\alpha$ queda determinada la solución exterior

$$ U_{out}(x) = \alpha + x, $$

que es justamente la parte "lineal" de la fórmula exacta de más arriba. El problema es que $U_{out}(1) = \alpha+1$, mientras que el dato de borde pide $U(1)=\beta$. Queda un desfasaje

$$ \beta - U_{out}(1) = \beta-\alpha-1 $$

que la solución exterior no puede cubrir porque ya usó su única constante libre en $x=0$. Ese desfasaje es exactamente el factor $(\beta-\alpha-1)$ que multiplica al término exponencial en $U_\varepsilon(x)$: la capa límite existe para "meter" esa corrección faltante en una zona angosta cerca de $x=1$.

A medida que $\varepsilon$ disminuye, la solución desarrolla una transición cada vez más abrupta cerca del borde $x=1$: es la **capa límite**, una región angosta donde la solución cambia muy rápido mientras que en el resto del dominio se mantiene casi constante.

### Resolución numérica

Discretizando con diferencias centradas tanto la derivada primera como la segunda sobre una malla de tamaño $h$, se obtiene un sistema lineal tridiagonal para los valores interiores de $u$:

In [ ]:
function capa_limite(f,N;ε=0.3,α=1,β=3)

    #completar
end

f(x) = -1.0

Comparamos la solución numérica con la solución exacta para $\varepsilon = 0.1$ y $N=100$:

In [ ]:
ε=0.1
N=100
x=0:1/N:1
U=capa_limite(f,N,ε=ε)
plot(x,U,label="solucion numerica")
plot!(x,U_ε.(x,ε),label="solucion exacta")

**Conclusión:** la aproximación numérica reproduce bien tanto la zona suave como la capa límite cerca de $x=1$, siempre que la malla sea suficientemente fina en relación a $\varepsilon$. Si $h \gg 2\varepsilon$, el esquema deja de resolver correctamente la capa límite y aparecen oscilaciones espurias — se puede verificar variando `N` y `ε` en la celda anterior.

Además de comparar la solución numérica con la exacta para un único par $(N,\varepsilon)$, conviene chequear dos cosas por separado:

1. que el esquema efectivamente converge con **orden 2** en $h$ cuando la malla resuelve bien la capa límite, y
2. qué pasa cuando la malla es demasiado gruesa respecto de $\varepsilon$: de dónde sale exactamente la condición $h \lesssim 2\varepsilon$ mencionada en la conclusión.

In [ ]:
ε = 0.1
Ns = [10, 20, 40, 80, 160, 320, 640]
hs = 1 ./ Ns
errores = Float64[]

for N in Ns
    h = 1/N
    x = 0:h:1
    U = capa_limite(f, N, ε=ε)
    Uexacta = U_ε.(x, ε)
    push!(errores, maximum(abs.(U .- Uexacta)))
end

orden = fill(NaN, length(errores))
for i in 2:length(errores)
    orden[i] = log(errores[i-1]/errores[i]) / log(hs[i-1]/hs[i])
end

for i in 1:length(Ns)
    println("N=$(Ns[i])   h=$(round(hs[i],digits=5))   error=$(round(errores[i],sigdigits=4))   orden≈$(round(orden[i],digits=2))")
end

plot(hs, errores, xscale=:log10, yscale=:log10, marker=:circle, label="error numérico", xlabel="h", ylabel="error máximo")
plot!(hs, hs.^2, linestyle=:dash, label="O(h²)")

Con $\varepsilon=0.1$ fijo, todos los $h$ usados cumplen $h<2\varepsilon=0.2$, así que la capa límite está siempre bien resuelta: el orden estimado debería estabilizarse cerca de **2**, consistente con el uso de diferencias centradas tanto para $U_x$ como para $U_{xx}$.

#### De dónde sale la condición $h \lesssim 2\varepsilon$

Mirando los coeficientes de la matriz tridiagonal armada en `capa_limite`, la entrada que multiplica a $U_{i+1}$ en la fila $i$ es

$$ \frac{\varepsilon}{h^2} - \frac{1}{2h}. $$

Para que el esquema respete un **principio del máximo discreto** (y por lo tanto no genere oscilaciones espurias), todas las entradas fuera de la diagonal deben tener el mismo signo que la de $U_{i-1}$, que es siempre positiva. Eso exige

$$ \frac{\varepsilon}{h^2} - \frac{1}{2h} \ge 0 \quad \Longleftrightarrow \quad h \le 2\varepsilon, $$

que es exactamente la condición mencionada en la conclusión de más arriba (a veces llamada condición de **número de Péclet de malla** $Pe_h = h/(2\varepsilon) \le 1$). Cuando $h>2\varepsilon$ esa entrada cambia de signo y aparecen las oscilaciones que se ven en la prueba siguiente.

In [ ]:
ε = 0.02
Ns_osc = [10, 50]   # h=0.1 (>2ε) vs h=0.02 (<2ε)
x_fino = 0:0.001:1

plt = plot(x_fino, U_ε.(x_fino, ε), label="solución exacta", linewidth=2, color=:black)
for N in Ns_osc
    h = 1/N
    x = 0:h:1
    U = capa_limite(f, N, ε=ε)
    estado = h < 2ε ? "sin oscilar (h<2ε)" : "oscila (h>2ε)"
    plot!(plt, x, U, marker=:circle, label="N=$N, h=$(round(h,digits=3)) — $estado")
end
plt

**Conclusión de las pruebas:** con $\varepsilon=0.02$ el umbral teórico es $h < 2\varepsilon = 0.04$. Con $N=10$ ($h=0.1$) el esquema oscila visiblemente cerca de $x=1$; con $N=50$ ($h=0.02$) la solución numérica ya resuelve la capa límite sin oscilaciones espurias — confirmando numéricamente la condición de número de Péclet de malla derivada arriba.

---

## Ecuación del calor: método explícito

Hasta acá trabajamos con problemas de contorno estacionarios. Ahora sumamos la variable temporal y consideramos la **ecuación del calor**

$$ u_t = u_{xx}, \qquad u(0,t)=u(1,t)=0, \qquad u(x,0)=U_0(x). $$

Una estrategia es pensar en un problema **semidiscro**: discretizamos solo en espacio con diferencias centradas y dejamos $t$ continuo. Eso convierte la EDP en un sistema de EDOs

$$ u_t = \frac{1}{h^2} A\, u, \qquad A = \text{Tridiagonal}(1,-2,1), $$

que es exactamente el tipo de sistema que ya sabemos integrar con los métodos de la clase anterior (Euler explícito, implícito, etc.). El código de abajo usa **Euler explícito** en el tiempo: un paso es $u^{n+1} = (I + rA)u^n$, con $r = k/h^2$.

Como $A$ es simétrica, sus autovalores son reales y conocidos en forma cerrada: $\lambda_j = -4\sin^2\!\left(\frac{j\pi h}{2}\right) \in [-4,0]$. La región de estabilidad de Euler explícito exige $|1+r\lambda_j|\le 1$ para todo autovalor, lo que da la condición

$$ r = \frac{k}{h^2} \le \frac{1}{2} \qquad\Longleftrightarrow\qquad k \le \frac{h^2}{2}, $$

A diferencia de una EDO de primer orden en el tiempo, acá el paso temporal permitido decae con $h^2$: mucho más restrictivo que en problemas hiperbólicos, donde suele alcanzar con $k=O(h)$.

In [ ]:
Uo(x) = sin(pi*x)
#Uo(x) =x*(1-x)

function calor_explicito(h,k,Uo)
  #completar 
  #debe retornar un grafico donde muestre la evolucion de los pasos.
end

calor_explicito (generic function with 1 method)

Probamos con $h=0.05$ y $k=0.0012$ (entonces $r=k/h^2=0.48<0.5$, régimen estable) y graficamos varios pasos superpuestos para ver cómo se va suavizando el perfil inicial:

In [ ]:
h = 0.05
k = 0.0012
p = calor_explicito(h,k,Uo)
display(p)

Para visualizar mejor la evolución conviene animarla en vez de superponer curvas. La siguiente función guarda todos los pasos en una matriz:

In [ ]:
function calor_explicito_anim(h,k,Uo)
  #completar
  sol = zeros(N+2,pasos)
  sol[:,1] = [0;u;0]
  for i=2:pasos
    u = M*u
    sol[:,i] = [0;u;0]
  end
  return sol
end

calor_explicito_anim (generic function with 1 method)

In [ ]:
u = calor_explicito_anim(h,k,Uo)
anim = @animate for n=1:100
  x=0:h:1
  plot(x,u[:,n],ylim=[0,1])
end
gif(anim, "calor.gif", fps = 10)

### Prueba numérica: ¿qué pasa si dejamos $\Delta t$ fijo y refinamos la malla?

Fijamos $k=\Delta t = 0.0013$ y probamos varios $h=\Delta x$. Como $U_0(x)=\sin(\pi x)$ es autofunción de $\partial_{xx}$ con estas condiciones de borde, la solución exacta de la ecuación del calor es

$$ u(x,t) = \sin(\pi x)\, e^{-\pi^2 t}, $$

así que podemos calcular el error contra la solución exacta en cualquier tiempo $T$ y ver qué pasa a medida que $h$ se achica con $k$ fijo.

In [ ]:
# Error: dejamos Δt fijo y variamos Δx
delta_t = 0.0013
delta_x = [0.05,0.06,0.07,0.08,0.09]
pasos = 100   # T final = pasos*delta_t

u_exacta(x,t) = sin(pi*x)*exp(-pi^2*t)

errores = Float64[]
rs = Float64[]

for h in delta_x
    x = 0:h:1
    r = delta_t/h^2
    push!(rs, r)
    N = length(x)-2
    A = Tridiagonal(ones(N-1),-2*ones(N),ones(N-1))
    M = Matrix(I,N,N) + r*A
    u = Uo.(x)[2:end-1]
    for i in 1:pasos
        u = M*u
    end
    T = pasos*delta_t
    Uex = u_exacta.(x[2:end-1], T)
    push!(errores, maximum(abs.(u .- Uex)))
end

for i in 1:length(delta_x)
    estado = rs[i] >= 0.5 ? "INESTABLE" : "estable"
    println("h=$(delta_x[i])   r=$(round(rs[i],digits=3))   error=$(errores[i])   ($estado)")
end

**Conclusión:** con $k$ fijo, refinar la malla (bajar $h$) *no* garantiza menor error. En cuanto $r=k/h^2$ cruza el umbral $1/2$ — acá pasa en $h=0.05$ — el esquema se vuelve inestable y el error explota, aunque la malla sea más fina. Esto muestra por qué la condición de estabilidad $k\le h^2/2$ no es un detalle técnico: si uno refina el espacio sin ajustar el paso temporal en consecuencia, se rompe la estabilidad.

### Orden de convergencia del método explícito

El error del esquema combina truncamiento espacial ($O(h^2)$, por las diferencias centradas) y temporal ($O(k)$, por Euler explícito), así que en general se espera $O(k+h^2)$. Pero acá no podemos fijar $k$ y solo refinar $h$ como en la prueba anterior, porque $r=k/h^2$ se dispararía y el esquema se volvería inestable antes de poder medir nada.

La forma correcta es refinar **espacio y tiempo juntos**, manteniendo $r=k/h^2$ fijo en la zona estable (por ejemplo $r=0.4$). Así $k$ queda atado a $h$ como $k=r\,h^2$, el error total se vuelve $O(h^2)$ (el término temporal $O(k)=O(h^2)$ queda del mismo orden que el espacial), y podemos estimar el orden con el mismo enfoque de razón local que usamos en la clase anterior.

In [ ]:
# Orden de convergencia: refinamos h y k juntos, con r=k/h^2 fijo (zona estable)
r0 = 0.4
T = 0.1
hs = [0.1, 0.05, 0.025, 0.0125, 0.00625]

errores = Float64[]
for h in hs
    k = r0*h^2
    pasos = round(Int, T/k)
    x = 0:h:1
    N = length(x)-2
    A = Tridiagonal(ones(N-1),-2*ones(N),ones(N-1))
    M = Matrix(I,N,N) + r0*A
    u = Uo.(x)[2:end-1]
    for i in 1:pasos
        u = M*u
    end
    Tfinal = pasos*k
    Uex = u_exacta.(x[2:end-1], Tfinal)
    push!(errores, maximum(abs.(u .- Uex)))
end

orden = fill(NaN, length(errores))
for i in 2:length(errores)
    orden[i] = log(errores[i-1]/errores[i]) / log(hs[i-1]/hs[i])
end

for i in 1:length(hs)
    println("h=$(hs[i])   k=$(round(r0*hs[i]^2,sigdigits=4))   error=$(round(errores[i],sigdigits=4))   orden≈$(round(orden[i],digits=2))")
end

plot(hs, errores, xscale=:log10, yscale=:log10, marker=:circle, label="error numérico", xlabel="h", ylabel="error máximo")
plot!(hs, hs.^2, linestyle=:dash, label="O(h²)")

**Se espera** que el orden estimado se acerque a **2**: al mantener $r$ fijo, tanto el error espacial como el temporal escalan como $h^2$, así que el esquema —a pesar de usar Euler explícito, que por sí solo es de orden 1 en el tiempo— termina convergiendo con orden 2 global, porque el paso temporal se refina "gratis" al doble de rápido que el espacial.